Stencil system

Install the stencil_lib wheel using pip 

In [81]:
import subprocess, sys, glob, pathlib

# location of .whl file
_search_paths = [
    pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path("."),
    pathlib.Path("../../build/dist"),
]
_wheel = next(
    (str(w) for p in _search_paths for w in p.glob("stencil_lib-*.whl")),
    None
)
if _wheel is None:
    raise FileNotFoundError("stencil_lib wheel not found. Run 'inv build' or place the wheel alongside the notebook.")

# pip install the wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       _wheel, "--force-reinstall"])


print(f"stencil_lib installed from: {_wheel}")


stencil_lib installed from: ..\..\build\dist\stencil_lib-0.1.0-py3-none-any.whl


Start using the stencil_lib library

In [82]:
# Start using the library
from stencil_lib import CipherConfig

In [83]:
class AESConfig(CipherConfig):
    algo = "aes"
    def __init__(self, key: bytes, mode: int, **mode_params):
        self.parameters = {
            "key": key,
            "mode": mode,
            "mode_params": mode_params
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.encrypt(plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.decrypt(ciphertext)


In [84]:
class CaesarConfig(CipherConfig):
    algo = "caesar"
    def __init__(self, shift: int):
        self.parameters = {
            "shift": shift
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b + shift) % 256 for b in plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b - shift) % 256 for b in ciphertext)


In [85]:
class VigenereConfig(CipherConfig):
    algo = "vigenere"
    def __init__(self, keyword: bytes):
        self.parameters = {
            "keyword": keyword
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b + key[i % key_len]) % 256 for i, b in enumerate(plaintext))
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b - key[i % key_len]) % 256 for i, b in enumerate(ciphertext))


# Stencil_system demo

The stencil library renders 3 APIs namely keygen, encrypt and decrypt

The usage of these APIs are demonstarated below.

In [ ]:
import random
import stencil_lib

grid_size = stencil_lib.GridShape(rows=12, cols=12)
in_byte_len = 16
plaintext = random.randbytes(in_byte_len)
num_partitions = 4
print(f"Plaintext : {' '.join(f'{b:02x}' for b in plaintext)}")

# cipher_cfg = AESConfig(key=random.randbytes(16), mode=1, iv=random.randbytes(16))
# cipher_cfg = CaesarConfig(shift=13)
cipher_cfg = VigenereConfig(keyword=b"KEY")

Plaintext : 40 79 b8 4d 7b 49 18 bf 0b fb 92 f0 c6 c6 d8 5a


In [ ]:
grid = stencil_lib.generate_random_grid(grid_size.rows, grid_size.cols)

print("Initial Grid:")
for i, row in enumerate(grid):
    hex_bytes = " ".join(f"{b:02x}" for b in row)
    print(f"  row {i:2d}: {hex_bytes}")


Initial Grid:
  row  0: 1b 56 c8 25 82 df 8d 12 5e b7 b6 58
  row  1: 19 51 be 78 4b d3 e3 2b 70 3c ae 37
  row  2: 7c 86 bc fa 57 c9 2a 38 8e dc 37 bf
  row  3: 4a d5 f4 83 36 64 b5 fe 23 e5 7c 33
  row  4: f5 47 74 03 1a 51 b2 06 48 24 dc 52
  row  5: a7 0b 9b fb fb 1d aa ef 14 b3 88 6b
  row  6: 65 1c fc 6e 66 f2 8f db f3 ed 3f 41
  row  7: 62 58 ff cd 9f a3 ca b0 82 33 98 71
  row  8: 78 5c 86 9e 42 35 9b d0 c0 2f 41 b0
  row  9: 15 c5 2d 37 2a a9 75 83 0b f7 5d 3d
  row 10: 42 0e d4 40 e1 0a ef ec a1 fb 17 cc
  row 11: 57 01 b3 a4 53 eb 8b 20 6b 47 ee c9


Keygen API

returns Secret key := stencil_lib.SecretKey type

In [ ]:
from stencil_lib import StencilConfig
stencil_cfg = StencilConfig(
    total_bytes=in_byte_len,
    num_partitions=num_partitions,
    grid_shape=grid_size,
    # grid_size is a GridShape instance in this notebook
    # (rename grid_size to grid_shape everywhere for consistency if you want)
    # or just pass grid_size as grid_shape
    # grid_shape=grid_shape,
    # ...
    # You may need to update variable names above if you change them
    # ...
)
secret_key = stencil_lib.keygen(
    stencil_cfg,
    cipher_cfg=cipher_cfg,
)

print("Secret Key")
print(f"  Cipher    : {secret_key.cipher_cfg.algo}")
print(f"  Partitions: {secret_key.partition_list}  ({len(secret_key.partition_list)} total, sum={sum(secret_key.partition_list)} bytes)")
print(f"  Stencils  : {len(secret_key.stencils)}")
for i, s in enumerate(secret_key.stencils):
    coords_str = ", ".join(f"({r},{c})" for r, c in s.coords)
    print(f"    [{i}] shape={s.shape!r}  len={s.len}  coords=[{coords_str}]")


Secret Key
  Cipher    : vigenere
  Partitions: [3, 11, 1, 1]  (4 total, sum=16 bytes)
  Stencils  : 4
    [0] shape='skewconnected'  len=3  coords=[(10,2), (9,2), (9,3)]
    [1] shape='skewconnected'  len=11  coords=[(2,8), (3,9), (3,10), (2,9), (1,10), (0,11), (0,10), (1,9), (2,10), (1,11), (2,11)]
    [2] shape='skewconnected'  len=1  coords=[(5,7)]
    [3] shape='skewconnected'  len=1  coords=[(9,10)]


Encrypt API 

returns obfuscated_grid := stencil_lib.Grid type

In [ ]:
obfuscated_grid = stencil_lib.encrypt(plaintext, secret_key, grid_size, grid)
print("\nWith Encryption API call, (Cipher text embedded) Obfuscated Grid is ready")



With Encryption API call, Cipher text embedded Obfuscated Grid ready


Print Cipher text and Obfuscated grid for Demo purpose:

In [95]:
from IPython.display import display, HTML

# Just for Demo purpose
ciphertext = cipher_cfg.encrypt(plaintext)

print(f"Ciphertext: {' '.join(f'{b:02x}' for b in ciphertext)}")

PART_COLORS = ["#f0a500", "#4fc3f7", "#81c784", "#f06292", "#ce93d8", "#80cbc4"]

# --- Partitioned ciphertext ---
parts_lines = []
offset = 0
for i, length in enumerate(secret_key.partition_list):
    chunk = ciphertext[offset: offset + length]
    color = PART_COLORS[i % len(PART_COLORS)]
    hex_part = " ".join(f"{b:02x}" for b in chunk)
    parts_lines.append(
        f'  P{i} ({length:2d}B): <span style="color:{color};font-weight:bold">{hex_part}</span>'
    )
    offset += length

display(HTML(
    "<b>Ciphertext — by partition</b>"
    '<pre style="line-height:1.8">' + "\n".join(parts_lines) + "</pre>"
))

# --- Obfuscated grid: all bytes of a partition share its color; first byte underlined ---
coord_to_part  = {(r, c): i for i, s in enumerate(secret_key.stencils) for r, c in s.coords}
first_coords   = {s.coords[0] for s in secret_key.stencils}

rows_html = []
for i, row in enumerate(obfuscated_grid):
    cells = []
    for j, b in enumerate(row):
        token = f"{b:02x}"
        if (i, j) in coord_to_part:
            part_idx = coord_to_part[(i, j)]
            color = PART_COLORS[part_idx % len(PART_COLORS)]
            underline = ";text-decoration:underline" if (i, j) in first_coords else ""
            token = f'<span style="color:{color};font-weight:bold{underline}">{token}</span>'
        cells.append(token)
    rows_html.append(f'  row {i:2d}: {" ".join(cells)}')

legend = "  ".join(
    f'<span style="color:{PART_COLORS[i % len(PART_COLORS)]};font-weight:bold">P{i}</span>'
    for i in range(len(secret_key.stencils))
)
display(HTML(
    f"<b>Obfuscated Grid</b> — {legend} (underlined = partition start)<br>"
    '<pre style="line-height:1.6">' + "\n".join(rows_html) + "</pre>"
))


Ciphertext: 8b be 11 98 c0 a2 63 04 64 46 d7 49 11 0b 31 a5


Decryprt API 

returns plaintext := bytes type

In [91]:
recovered = stencil_lib.decrypt(obfuscated_grid, secret_key)
assert recovered == plaintext

print(f"Decrypted Plaintext : {' '.join(f'{b:02x}' for b in recovered)}")

Decrypted Plaintext : 40 79 b8 4d 7b 49 18 bf 0b fb 92 f0 c6 c6 d8 5a
